# Lab | Web Scraping

Welcome to the "Books to Scrape" Web Scraping Adventure Lab!

**Objective**

In this lab, we will embark on a mission to unearth valuable insights from the data available on Books to Scrape, an online platform showcasing a wide variety of books. As data analyst, you have been tasked with scraping a specific subset of book data from Books to Scrape to assist publishing companies in understanding the landscape of highly-rated books across different genres. Your insights will help shape future book marketing strategies and publishing decisions.

**Background**

In a world where data has become the new currency, businesses are leveraging big data to make informed decisions that drive success and profitability. The publishing industry, much like others, utilizes data analytics to understand market trends, reader preferences, and the performance of books based on factors such as genre, author, and ratings. Books to Scrape serves as a rich source of such data, offering detailed information about a diverse range of books, making it an ideal platform for extracting insights to aid in informed decision-making within the literary world.

**Task**

Your task is to create a Python script using BeautifulSoup and pandas to scrape Books to Scrape book data, focusing on book ratings and genres. The script should be able to filter books with ratings above a certain threshold and in specific genres. Additionally, the script should structure the scraped data in a tabular format using pandas for further analysis.

**Expected Outcome**

A function named `scrape_books` that takes two parameters: `min_rating` and `max_price`. The function should scrape book data from the "Books to Scrape" website and return a `pandas` DataFrame with the following columns:

**Expected Outcome**

- A function named `scrape_books` that takes two parameters: `min_rating` and `max_price`.
- The function should return a DataFrame with the following columns:
  - **UPC**: The Universal Product Code (UPC) of the book.
  - **Title**: The title of the book.
  - **Price (£)**: The price of the book in pounds.
  - **Rating**: The rating of the book (1-5 stars).
  - **Genre**: The genre of the book.
  - **Availability**: Whether the book is in stock or not.
  - **Description**: A brief description or product description of the book (if available).
  
You will execute this script to scrape data for books with a minimum rating of `4.0 and above` and a maximum price of `£20`. 

Remember to experiment with different ratings and prices to ensure your code is versatile and can handle various searches effectively!

**Resources**

- [Beautiful Soup Documentation](https://www.crummy.com/software/BeautifulSoup/bs4/doc/)
- [Pandas Documentation](https://pandas.pydata.org/pandas-docs/stable/index.html)
- [Books to Scrape](https://books.toscrape.com/)


**Hint**

Your first mission is to familiarize yourself with the **Books to Scrape** website. Navigate to [Books to Scrape](http://books.toscrape.com/) and explore the available books to understand their layout and structure. 

Next, think about how you can set parameters for your data extraction:

- **Minimum Rating**: Focus on books with a rating of 4.0 and above.
- **Maximum Price**: Filter for books priced up to £20.

After reviewing the site, you can construct a plan for scraping relevant data. Pay attention to the details displayed for each book, including the title, price, rating, and availability. This will help you identify the correct HTML elements to target with your scraping script.

Make sure to build your scraping URL and logic based on the patterns you observe in the HTML structure of the book listings!


---

**Best of luck! Immerse yourself in the world of books, and may the data be with you!**

**Important Note**:

In the fast-changing online world, websites often update and change their structures. When you try this lab, the **Books to Scrape** website might differ from what you expect.

If you encounter issues due to these changes, like new rules or obstacles preventing data extraction, don’t worry! Get creative.

You can choose another website that interests you and is suitable for scraping data. Options like Wikipedia, The New York Times, or even library databases are great alternatives. The main goal remains the same: extract useful data and enhance your web scraping skills while exploring a source of information you enjoy. This is your opportunity to practice and adapt to different web environments!

In [ ]:
import re
import time
import requests
import pandas as pd
from bs4 import BeautifulSoup

BASE_URL = "http://books.toscrape.com/catalogue/"
RATING_WORDS = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}


def get_soup(url):
    """Descarga una página y la convierte en un objeto BeautifulSoup."""
    response = requests.get(url, timeout=20)
    response.encoding = "utf-8"          # para que el símbolo £ se lea bien
    return BeautifulSoup(response.text, "html.parser")


def parse_book_page(url):
    """Entra en la ficha de un libro y saca UPC, género, disponibilidad y descripción."""
    soup = get_soup(url)
    table = {row.th.text.strip(): row.td.text.strip() for row in soup.select("table.table tr")}
    genre = soup.select("ul.breadcrumb li a")[2].text.strip()
    desc_tag = soup.select_one("#product_description + p")
    return {
        "UPC": table.get("UPC"),
        "Genre": genre,
        "Availability": table.get("Availability"),
        "Description": desc_tag.text.strip() if desc_tag else None,
    }


def scrape_books(min_rating, max_price):
    """Recorre las 50 páginas del catálogo y devuelve un DataFrame con los libros
    que tienen rating >= min_rating y precio <= max_price."""
    books = []
    page_url = BASE_URL + "page-1.html"

    while page_url:
        soup = get_soup(page_url)

        for article in soup.select("article.product_pod"):
            # Datos que ya están en el listado: título, precio y rating
            title = article.h3.a["title"]
            price = float(re.sub(r"[^0-9.]", "", article.select_one("p.price_color").text))
            rating_word = article.select_one("p.star-rating")["class"][1]
            rating = RATING_WORDS[rating_word]

            # Filtramos ANTES de entrar en la ficha para no hacer peticiones innecesarias
            if rating >= min_rating and price <= max_price:
                book_url = BASE_URL + article.h3.a["href"]
                details = parse_book_page(book_url)
                books.append({
                    "UPC": details["UPC"],
                    "Title": title,
                    "Price (£)": price,
                    "Rating": rating,
                    "Genre": details["Genre"],
                    "Availability": details["Availability"],
                    "Description": details["Description"],
                })
                time.sleep(0.1)   # ser educados con el servidor

        # ¿Hay página siguiente?
        next_link = soup.select_one("li.next a")
        page_url = BASE_URL + next_link["href"] if next_link else None

    return pd.DataFrame(books)

In [ ]:
# Ejecutar con rating mínimo 4 y precio máximo 20 £ (tarda 1-2 minutos)
df_books = scrape_books(min_rating=4, max_price=20)
print(df_books.shape)
df_books.head()

In [ ]:
# Un poco de análisis: cuántos libros y precio medio por género
summary = (df_books.groupby("Genre")
           .agg(books=("Title", "count"), avg_price=("Price (£)", "mean"))
           .sort_values("books", ascending=False)
           .round(2))
summary.head(10)

**Conclusiones:** la tabla muestra en qué géneros hay más libros bien valorados (4 y 5 estrellas) por menos de 20 £. Esos géneros son los candidatos para campañas de marketing de "alta calidad a buen precio". Los géneros con pocos libros en el filtro son nichos donde la competencia en esa franja de precio es menor.